In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

In [3]:
df = pd.read_csv('gurgaon_properties_13k_outlier_treated.csv')
df.shape

(13191, 23)

In [4]:
df['facing'].value_counts(dropna=False)

facing
North-East    3900
East          2859
NaN           2106
North         1692
West           804
North-West     523
South-East     500
South          452
South-West     355
Name: count, dtype: int64

In [5]:
df['facing'].isnull().sum()

np.int64(2106)

In [6]:
df['facing'].isnull().sum()

np.int64(2106)

In [7]:
# Load raw gurgaon_properties 13k.csv to retrieve the 'description' column
raw_df = pd.read_csv('gurgaon_properties 13k.csv')
raw_df.head(2)

,property_name,property_type,society,price,price_per_sqft,area,areaWithType,bedRoom,bathroom,balcony,additionalRoom,address,floorNum,facing,agePossession,nearbyLocations,description,furnishDetails,features,rating
0,8 Bedroom House for sale in Ansal Plaza,house,independent,5.5,21825.0,2520.0,Plot area 280(234.12 sq.m.),8,8,2,"pooja room,study room,servant room","Independent House, Ansal Plaza, Gurgaon, Haryana",3.0,East,5 to 10 Year Old,"['HUDA Sector 23 Market', 'Palam triangle', 'Palam Vihar Vyapar kendra', 'Ram Mandir', ""DR KAPOOR'S Dental Care and Implant Centre"", 'Metro Hospital and Heart Institute Gurgaon', 'Kalyan Hospital Gurgaon', 'Ansals plaza underground car parking', 'Ansals Plaza above ground car parking', 'HUDA sector 23 parking', 'Big Cinemas', 'Axis bank', 'Punjab national bank', 'Oriental bank of commerce', 'Moti Mahal', 'Pind Baluchi', 'Pizza Hut', ""Domino's Pizza"", 'Cafe Coffee Day', 'Om Sweets', ""McDonald's"", 'Masti ki Pathshala Teach India', 'Swiss Cottage school', 'Chiranjiv Bharati School', 'THE NORTHCAP UNIVERSITY', 'Palam vihar railway station', 'Bijwasan railway station']","This home is situated in a prime and peaceful neighborhood, and is close to the market, mall and hospital.About locality -The locality is close to nh-48 thus having excellent connectivity with new delhi\nIgi airport - 12 km (25 minutes)\nNearest metro stations -\n1) mg road metro station - Yellow line - 6.5 km (15 minutes)\n2) dwarka sector 21 metro station blue line - 7 km (15 minutes)\nDlf cyberhub and mgf metropolitan mall - 6.5 km (15 minutes)\nEmployment hubs of gurgaon - Udyog vihar and cyber city are also 4 km away (12-15 minutes)Nearby attractions - Sector 22 market, sector 23 market, ansal plaza mall, global foyer mall, palam vihar vyapar kendra, columbia asia hospital, metro hospital, sector 22 market, 24 seven store (Sec-22)Some major restaurants within 500m - Pizza hut, dominos, mcdonalds, om sweets, haldirams, barbeque nation, tossin pizza, subway, nirula's, baskin robbins, giani's, etc.","['8 Wardrobe', '11 Fan', '1 Exhaust Fan', '18 Light', '1 Chimney', '1 Modular Kitchen', 'No AC', 'No Bed', 'No Curtains', 'No Dining Table', 'No Geyser', 'No Microwave', 'No Fridge', 'No Sofa', 'No Stove', 'No TV', 'No Washing Machine', 'No Water Purifier']","['Feng Shui / Vaastu Compliant', 'Private Garden / Terrace', 'High Ceiling Height', 'False Ceiling Lighting', 'Water Storage', 'Separate entry for servant room', 'No open drainage around', 'Piped-gas', 'Park', 'Natural Light', 'Spacious Interiors', 'Fitness Centre / GYM', 'Waste Disposal', 'Club house / Community Center']","['Environment5 out of 5', 'Safety5 out of 5', 'Lifestyle5 out of 5', 'Connectivity5 out of 5']"
1,3 BHK Flat in Sector 83 Gurgaon,flat,emaar palm gardens,1.8,9473.0,1900.0,Super Built up area 1900(176.52 sq.m.)Carpet area: 1240.04 sq.ft. (115.2 sq.m.),3,3,3,servant room,"605, Sector 83 Gurgaon, Gurgaon, Haryana",6.0,South,1 to 5 Year Old,"[""V'Lante Mall"", 'NH 48 Gurugram', 'iGrow Montessori Play School', 'Matrikiran High School', 'Badsa AMS Hospital', 'GlobalHealthcare Multispeciality', 'Indira Gandhi International Airport', 'Patli Railway Station', 'Hyatt Regency', 'Nakhrola Stadium', 'Cricket Academy']","This lovely 3 bhk apartment/flat in sector 83 gurgaon is available for sale in one of gurgaon's most popular projects, emaar palm gardens. The flat is west-Facing. Containing 3 bedroom(s), 3 bathrooms and 3 balconies, this flat is spread over a super built up area of 1900 sq.Ft. The flat has a total of 16 floors and this property is situated on 6th floor. As the project is already ready to move, so you can easily move into this 1-5 years old property. The flat comes well built with vitrified flooring options. Proximity to landmarks like close to school, close to mall and close to hospital makes this an ideal property for families. All the modern amenities such as swimming pool, grocery shop, club house / community center, fitness centre / gym, park, 

In [8]:
# Pass 1: Match on all 9 shared property attributes
shared_cols_1 = [
    'property_type', 'society', 'price', 'price_per_sqft',
    'areaWithType', 'bedRoom', 'bathroom', 'balcony', 'floorNum'
]

# Pass 2: Fallback match excluding 'price_per_sqft' (for rows updated during sq.yard conversion)
shared_cols_2 = [
    'property_type', 'society', 'price',
    'areaWithType', 'bedRoom', 'bathroom', 'balcony', 'floorNum'
]

raw_sub_1 = raw_df.drop_duplicates(subset=shared_cols_1)[shared_cols_1 + ['description']]
raw_sub_2 = raw_df.drop_duplicates(subset=shared_cols_2)[shared_cols_2 + ['description']]

desc_1 = df.merge(raw_sub_1, on=shared_cols_1, how='left')['description']
desc_2 = df.merge(raw_sub_2, on=shared_cols_2, how='left')['description']

df['description'] = desc_1.fillna(desc_2).values
print("Missing descriptions:", df['description'].isna().sum())

Missing descriptions: 0


In [9]:
df[['society', 'facing', 'description']].sample(5, random_state=42)

,society,facing,description
5282,bptp pedestal,North-East,"Carefully laid out in the prime location of gurgaon, this elegant 4 bhk builder floor in sector 70a gurgaon is a well-Suited property for you. This builder floor comes with 4 bedrooms, 3 bathrooms and 3 balconies. This property is furnished. This builder floor provides you carpet area of 2100 sq.Ft. This builder floor is situated at the 2nd floor of 3 floors building. This 0-1 year old builder floor is ready to move in. It is a freehold property. The price of this property is 1.8 crore. Minor negotiations can be considered."
704,dlf regal gardens,East,"Dlf regal gardens sec-90.\nThis is a well - Designed 4 bhk resale apartment located in sector-90, gurgaon. It is a freehold property. The flat is a spacious property and is ready to move in. Located in an integrated society of dlf regal gardens, it has 4 bathroom(s) and more than 3 balcony(s) and 5 a.C (s). This is a feng shui/vaastu compliant property and has wood flooring. It offers a wonderful view of park/garden and main road. It is a 0-1 year old property, located on the 12th floor with 6 ac. Full power back up. It is a corner unit located in a gated society. The property offers specifications such as club house/community center, fitness centre/gym, intercom facility, lift(s), park, rain water harvesting, swimming pool, visitor parking, water storage, security/fire alarm, piped-Gas and water purifier. The apartment is approximately priced at rs. 13500000 and its per sq. Ft. Cost is rs. 6095. 1 covered parking is also available.\n \n \n \n \n Additional details :Piped gas facility is available in the property.\nThe apartment has borings water supply.\nDaily needs shopping could be done within the society premises to make the stay convinent.\nFull power backup is available.\nThere is also a separate washroom for domestic help.\nThe society has dedicated security guards for every tower."
9081,conscient elevate,NaN,"Situated in sector 59 gurgaon, conscient elevate is a well planned society that offers a pleasant living experience to its residents. This 3 bhk flat in gurgaon is your opportunity to be a part of this community"
6143,independent,East,"East facing renovated kothi available for sale in dlf 2, this house is in k block. Walking distance from metro station, malls , market , hospital. Loan available. Keys in hand"
8106,orchid petals,NaN,"Ready to move duplex penthouse -5 bhk+puja+sr unused available for \nIn orchid petals, sector 49, gurgaon.\nIt is a 37 acres of gated society in the main sohna elevated road, gurugram.\nThis society is a mixed of 2bhk+study/3bhk+study/4bhk+sr/duplex penthouse of 5bhk & villas also with the occupancy level of 95-98 percent.\n\nTotal 25 towers in this gated community and each tower has 14 apartments and 4 apartments per floor.\nNewly renovated entrance lobby of each towers.\n24x7 security with tower wise guard.\n24x7 gmda water supply\nPiped gas line/3 swimming pools/bedminton and basket ball court/gym/huge big green parks.\n2 levels of basement carparking.\n\nFor more information and site visit, please contact us."


In [10]:
# Stage 1: Function to extract explicit facing direction from 'description'
def extract_facing_from_desc(text):
    if not isinstance(text, str):
        return np.nan
    
    # 1a. Compound directions (North-East, North-West, South-East, South-West, NE, NW, SE, SW)
    m_comp = re.search(
        r'\b(north[\s-]*east|north[\s-]*west|south[\s-]*east|south[\s-]*west|ne|nw|se|sw)\s*'
        r'(?:-\s*)?(?:facing|face|direction|entry|entrance|property|flat|unit|side|corner|open)\b'
        r'|\b(?:facing|face|entry|entrance|direction)\s*(?:is\s*|towards\s*|to\s*|:?\s*)?'
        r'(north[\s-]*east|north[\s-]*west|south[\s-]*east|south[\s-]*west)\b'
        r'|\b(north[\s-]*east)\b(?=.*vastu|.*vaastu)',
        text, re.IGNORECASE
    )
    if m_comp:
        raw = (m_comp.group(1) or m_comp.group(2) or m_comp.group(3)).upper().replace(' ', '').replace('-', '')
        mapping = {
            'NORTHEAST': 'North-East', 'NE': 'North-East',
            'NORTHWEST': 'North-West', 'NW': 'North-West',
            'SOUTHEAST': 'South-East', 'SE': 'South-East',
            'SOUTHWEST': 'South-West', 'SW': 'South-West'
        }
        return mapping.get(raw, np.nan)
    
    # 1b. Cardinal directions (East, West, North, South)
    m_card = re.search(
        r'\b(east|west|north|south)(?:\s+and\s+park)?\s*(?:-\s*)?(?:facing|face|entry|entrance|direction)\b'
        r'|\b(?:facing|face|entry|entrance|exit\s+is)\s*(?:is\s*|towards\s*|to\s*|:?\s*)?(east|west|north|south)\b',
        text, re.IGNORECASE
    )
    if m_card:
        return (m_card.group(1) or m_card.group(2)).strip().title()
    
    # 1c. Domain conventions ('sun facing' -> East, 'vastu compliant' -> North-East)
    if re.search(r'\b(?:sun\s*facing|morning\s*sunlight|first\s*light\s*of\s*the\s*day)\b', text, re.IGNORECASE):
        return 'East'
    if re.search(r'\b(?:vastu|vaastu)\s*(?:compliant|complaint|perfect|compatible|friendly)\b', text, re.IGNORECASE):
        return 'North-East'
    
    return np.nan

In [11]:
# Apply Stage 1 extraction on missing 'facing' rows
mask_na = df['facing'].isna()
df.loc[mask_na, 'facing'] = df.loc[mask_na, 'description'].apply(extract_facing_from_desc)
df['facing'].isnull().sum()

np.int64(1998)

In [12]:
# Stage 2: Web-verified project orientation lookup (from 99acres & MagicBricks)
web_society_facing = {
    'signature global park 4': 'North-East',
    'silverglades the melia': 'North-East',
    'tulip monsella': 'North-East',
    'kendriya vihar': 'North',
    'signature global the millennia': 'East',
    'signature global the millennia 3': 'East',
    'ild grand': 'East',
    'pyramid elite': 'North-East',
    'unitech sunbreeze': 'North-East',
    'unitech vistas': 'East',
    'whiteland blissville': 'North-East',
    'pivotal riddhi siddhi': 'East',
    'sare green parc': 'North-East',
    'adani aangan arcade': 'East',
    'sare petioles': 'East',
    'sidhartha ncr greens': 'North-East',
    'global hill view': 'East',
    'tulsiani easy in homes': 'North-East',
    'vatika sovereign next': 'North-East',
    'suposhaa smartworld orchard': 'East',
    'nbcc heights': 'North-East',
    'm3m sky lofts': 'North-East',
    'adani aangan galleria': 'East',
    'unitech uniworld gardens 2': 'East',
    'birla navya avik': 'North',
    'mittal surya vihar': 'East',
    'adani brahma samsara phase 2': 'North',
    'irwo westend towers': 'South-East',
    'dlf the arbour': 'North-East'
}

soc_clean = df['society'].astype(str).str.strip().str.lower()
mask_web = df['facing'].isna() & soc_clean.isin(web_society_facing.keys())
df.loc[mask_web, 'facing'] = soc_clean[mask_web].map(web_society_facing)
df['facing'].isnull().sum()

np.int64(1904)

In [13]:
# Stage 3: Exact Floor-Plan Configuration Match (society + bedRoom + area)
config_mode = df.groupby(['society', 'bedRoom', 'area'])['facing'].transform(
    lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
)
df['facing'] = df['facing'].fillna(config_mode)
df['facing'].isnull().sum()

np.int64(997)

In [14]:
# Stage 4a: Society-level mode match
society_mode = df.groupby('society')['facing'].transform(
    lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
)
df['facing'] = df['facing'].fillna(society_mode)
df['facing'].isnull().sum()

np.int64(145)

In [15]:
# Stage 4b: (sector, property_type) mode match & Sector mode fallback
sector_prop_mode = df.groupby(['sector', 'property_type'])['facing'].transform(
    lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
)
df['facing'] = df['facing'].fillna(sector_prop_mode)

sector_mode = df.groupby('sector')['facing'].transform(
    lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
)
df['facing'] = df['facing'].fillna(sector_mode)
df['facing'] = df['facing'].fillna(df['facing'].mode().iloc[0])
df['facing'].isnull().sum()

np.int64(0)

In [16]:
df['facing'].value_counts(dropna=False)

facing
North-East    4957
East          3501
North         1982
West           833
North-West     542
South-East     526
South          479
South-West     371
Name: count, dtype: int64

In [17]:
df.drop(columns=['description'], inplace=True)

C:\Users\prama\AppData\Local\Temp\ipykernel_13832\3051626672.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(columns=['description'], inplace=True)


In [18]:
df.sample(5)

,property_type,society,sector,price,price_per_sqft,area,areaWithType,bedRoom,bathroom,balcony,floorNum,facing,agePossession,super_built_up_area,built_up_area,carpet_area,study room,servant room,store room,pooja room,others,furnishing_type,luxury_score
7229,flat,suncity avenue 102,sector 102,0.55,10617.0,518.0,Super Built up area 605.5(56.25 sq.m.)Carpet area: 518 sq.ft. (48.12 sq.m.),2,2,1,8.0,North-East,Relatively New,605.5,NaN,518.0,0,0,0,0,0,0,57
3631,flat,m3m heights,sector 65,1.71,13542.0,1263.0,Super Built up area 1265(117.52 sq.m.)Carpet area: 1261 sq.ft. (117.15 sq.m.),2,2,2,37.0,North-East,Under Construction,1265.0,NaN,1261.0,0,0,0,0,0,0,120
3319,flat,pioneer araya,sector 62,10.75,19500.0,5513.0,Super Built up area 5514(512.27 sq.m.),5,7,3,10.0,North-East,Relatively New,5514.0,NaN,NaN,0,1,0,0,0,1,189
8092,flat,chd avenue 71,sector 71,1.12,6481.0,1728.0,Super Built up area 1620(150.50 sq.m.),3,3,3,11.0,South,Relatively New,1620.0,NaN,NaN,1,0,0,0,0,2,156
6638,flat,mapsko paradise,sector 83,0.80,8000.0,1000.0,Super Built up area 1340(124.49 sq.m.)Carpet area: 1000 sq.ft. (92.90 sq.m.),3,3,3,1.0,North-East,Relatively New,1340.0,NaN,1000.0,1,0,0,0,0,2,89


In [19]:
df.shape

(13191, 23)

In [20]:
df.duplicated().sum()

np.int64(1)

In [21]:
df.drop_duplicates(inplace=True)

C:\Users\prama\AppData\Local\Temp\ipykernel_13832\3006716147.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop_duplicates is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop_duplicates(inplace=True)


In [22]:
df.to_csv('gurgaon_properties_13k_outlier_treated_facing_filled.csv', index=False)